In [ ]:
from pathlib import Path
import json

def resolve_assignment_dir():
    cwd = Path.cwd().resolve()

    # Local/GPU-lab checkout.
    for p in [cwd, *cwd.parents]:
        if p.name == "GPU_Assignment1" or (p / "README_FIRST.md").exists():
            return p

    # Google Colab / Google Drive.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        root = Path("/content/drive/MyDrive")
        matches = list(root.rglob("GPU_Assignment1"))
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(
                "Multiple GPU_Assignment1 folders found: "
                + ", ".join(str(x) for x in matches)
            )
    except ImportError:
        pass

    raise FileNotFoundError(
        "Could not locate GPU_Assignment1. Run from the assignment folder "
        "or mount Google Drive."
    )

ASSIGNMENT_DIR = resolve_assignment_dir()
ARTIFACT_DIR = ASSIGNMENT_DIR / "hw2_5_artifacts"
PARAM_FILE = ARTIFACT_DIR / "student_params.json"

if not PARAM_FILE.exists():
    raise FileNotFoundError(f"student_params.json not found at: {PARAM_FILE}")

PARAMS = json.loads(PARAM_FILE.read_text())
SID4 = PARAMS["SID4"]
SEED = PARAMS["SEED"]
SLICE = PARAMS["SLICE"]
HP_ID = PARAMS["HP_ID"]
CLS_A = PARAMS["CLS_A"]
CLS_B = PARAMS["CLS_B"]

print("Loaded:", PARAM_FILE)


# HW2.5 — Notebook 03
## Part E: Sustained load / thermal behavior
## Part F: Final analysis and submission artifacts

**Important:** the sustained-load cell intentionally runs for **20 minutes (1200 seconds)** and samples approximately every **5 seconds**, as required.

It logs:
- SM clock
- memory clock
- GPU temperature
- power draw
- utilization
- active clock-throttle reasons when the driver exposes them
- achieved matmul throughput for each sampling window

The final section reads Parts B–D outputs and produces `METRICS.md` plus an `AI_USE.md` template containing the exact four required prompts.

In [ ]:
import os, sys, gc, math, time, json, random, traceback, subprocess, platform, re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

FIG_DIR = ARTIFACT_DIR / "figures"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = ARTIFACT_DIR / "RUN_LOG.txt"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def log(message=""):
    line = str(message)
    print(line)
    with RUN_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{utc_now()}] {line}\n")

def run_cmd(cmd, check=False):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed: {cmd}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}"
        )
    return p

GPU_INFO_FILE = ARTIFACT_DIR / "gpu_info.json"
if not GPU_INFO_FILE.exists():
    raise FileNotFoundError("gpu_info.json missing. Run notebook 00 first.")

GPU_INFO = json.loads(GPU_INFO_FILE.read_text())
GPU_UUID = GPU_INFO["uuid"]
GPU_NAME = GPU_INFO["name"]
GPU_INDEX = int(GPU_INFO["nvidia_smi_index"])
CARD_SPECS = GPU_INFO["card_specs"]

if not GPU_INFO.get("hw25_supported_gpu", False):
    raise RuntimeError("Notebook 00 did not record an approved RTX 4090/5090 GPU.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

current = run_cmd([
    "nvidia-smi",
    "--query-gpu=index,name,uuid",
    "--format=csv,noheader,nounits"
], check=True)
current_rows = [[x.strip() for x in line.split(",")] for line in current.stdout.strip().splitlines()]
if not any(len(r) == 3 and r[2] == GPU_UUID for r in current_rows):
    raise RuntimeError(
        "The current GPU UUID does not match notebook 00. "
        "Rerun notebook 00 on this reserved GPU."
    )

DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

log("=" * 88)
log(
    f"Notebook started | GPU={GPU_NAME} | UUID={GPU_UUID} | "
    f"Driver CUDA={GPU_INFO.get('driver_cuda_version')} | "
    f"PyTorch CUDA build={torch.version.cuda}"
)


# Part E — 20-minute sustained compute load

The workload repeatedly performs FP16 `8192 × 8192` matrix multiplications. Each sample window is synchronized so throughput corresponds to completed GPU work.

Set `RUN_20_MIN_TEST = True` for your **final executed submission run**. It is already `True` below.

In [ ]:
RUN_20_MIN_TEST = True
DURATION_SECONDS = 20 * 60
FAST_PHASE_SECONDS = 30.0
FAST_SAMPLE_INTERVAL_SECONDS = 1.0
NORMAL_SAMPLE_INTERVAL_SECONDS = 5.0
LOAD_N = 8192
LOAD_DTYPE = torch.float16
MATMULS_PER_SYNC = 4

def parse_float(s):
    try:
        return float(str(s).strip())
    except Exception:
        return np.nan

BASE_FIELDS = [
    "timestamp", "uuid", "clocks.sm", "clocks.mem", "temperature.gpu",
    "power.draw", "utilization.gpu", "power.limit"
]
THROTTLE_FIELDS = [
    "clocks_throttle_reasons.active",
    "clocks_throttle_reasons.sw_power_cap",
    "clocks_throttle_reasons.hw_slowdown",
    "clocks_throttle_reasons.hw_thermal_slowdown",
    "clocks_throttle_reasons.hw_power_brake_slowdown",
]

def supported_smi_fields():
    all_fields = BASE_FIELDS + THROTTLE_FIELDS
    p = run_cmd([
        "nvidia-smi", "-i", str(GPU_INDEX),
        "--query-gpu=" + ",".join(all_fields),
        "--format=csv,noheader,nounits"
    ])
    if p.returncode == 0:
        return all_fields
    log("Some throttle-reason fields are not supported by this driver; using base telemetry only.")
    log("nvidia-smi probe stderr: " + p.stderr.strip())
    return BASE_FIELDS

SMI_FIELDS = supported_smi_fields()

def query_telemetry():
    p = run_cmd([
        "nvidia-smi", "-i", str(GPU_INDEX),
        "--query-gpu=" + ",".join(SMI_FIELDS),
        "--format=csv,noheader,nounits"
    ], check=True)
    vals = [x.strip() for x in p.stdout.strip().split(",")]
    if len(vals) != len(SMI_FIELDS):
        raise RuntimeError(f"Unexpected telemetry row: {p.stdout!r}")
    return dict(zip(SMI_FIELDS, vals))

thermal_path = ARTIFACT_DIR / "thermal_log.csv"

if RUN_20_MIN_TEST:
    gc.collect()
    torch.cuda.empty_cache()

    a = torch.randn((LOAD_N, LOAD_N), device=DEVICE, dtype=LOAD_DTYPE)
    b = torch.randn((LOAD_N, LOAD_N), device=DEVICE, dtype=LOAD_DTYPE)

    for _ in range(5):
        _ = torch.mm(a, b)
    torch.cuda.synchronize()

    rows = []
    wall_start = time.monotonic()

    log(
        f"Part E START | UUID={GPU_UUID} | duration={DURATION_SECONDS}s | "
        f"sampling=1s for first {FAST_PHASE_SECONDS:.0f}s, then 5s | "
        f"load=FP16 matmul N={LOAD_N}"
    )

    while True:
        elapsed_before = time.monotonic() - wall_start
        if elapsed_before >= DURATION_SECONDS:
            break

        sample_interval = (
            FAST_SAMPLE_INTERVAL_SECONDS
            if elapsed_before < FAST_PHASE_SECONDS
            else NORMAL_SAMPLE_INTERVAL_SECONDS
        )

        window_start = time.monotonic()
        iterations = 0

        while (
            (time.monotonic() - window_start) < sample_interval
            and (time.monotonic() - wall_start) < DURATION_SECONDS
        ):
            for _ in range(MATMULS_PER_SYNC):
                c = torch.mm(a, b)
                iterations += 1
            torch.cuda.synchronize()

        window_end = time.monotonic()
        window_seconds = window_end - window_start
        elapsed = window_end - wall_start

        tel = query_telemetry()
        tflops = (iterations * 2.0 * LOAD_N**3) / window_seconds / 1e12

        row = {
            "elapsed_s": elapsed,
            "window_s": window_seconds,
            "target_sample_interval_s": sample_interval,
            "matmul_iterations": iterations,
            "throughput_tflops": tflops,
            "uuid": GPU_UUID,
            "gpu": GPU_NAME,
            "sm_clock_mhz": parse_float(tel.get("clocks.sm")),
            "memory_clock_mhz": parse_float(tel.get("clocks.mem")),
            "temperature_C": parse_float(tel.get("temperature.gpu")),
            "power_draw_W": parse_float(tel.get("power.draw")),
            "utilization_pct": parse_float(tel.get("utilization.gpu")),
            "power_limit_W": parse_float(tel.get("power.limit")),
            "smi_timestamp": tel.get("timestamp", ""),
        }

        for f in THROTTLE_FIELDS:
            row[f] = tel.get(f, "unsupported")

        rows.append(row)
        pd.DataFrame(rows).to_csv(thermal_path, index=False)

        log(
            f"Part E sample | UUID={GPU_UUID} | t={elapsed:.1f}s | "
            f"clock={row['sm_clock_mhz']:.0f}MHz | temp={row['temperature_C']:.1f}C | "
            f"power={row['power_draw_W']:.1f}W | util={row['utilization_pct']:.0f}% | "
            f"throughput={tflops:.2f}TFLOPS"
        )

    del a, b, c
    gc.collect()
    torch.cuda.empty_cache()
    log(f"Part E END | wrote {thermal_path} | samples={len(rows)}")

thermal = pd.read_csv(thermal_path)
display(thermal.head())
display(thermal.tail())


## Part E.2 — plot clock and temperature against time on one figure

A second y-axis is used so clock and temperature remain readable on the same figure.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5.8))

line1 = ax1.plot(
    thermal["elapsed_s"],
    thermal["sm_clock_mhz"],
    color="tab:blue",
    label="SM clock (MHz)",
)
ax1.set_xlabel("Elapsed time (s)")
ax1.set_ylabel("SM clock (MHz)")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
line2 = ax2.plot(
    thermal["elapsed_s"],
    thermal["temperature_C"],
    color="tab:red",
    label="Temperature (°C)",
)
ax2.set_ylabel("Temperature (°C)")

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="best")
ax1.set_title(f"HW2.5 Part E — Sustained Load\n{GPU_NAME} | {GPU_UUID}")

fig.tight_layout()
p = FIG_DIR / "part_e_clock_temperature_vs_time.png"
fig.savefig(p, dpi=180)
plt.show()
log(f"Saved {p}")


## Part E.3 — peak vs steady-state throughput and throttle interpretation

- The first 30 seconds are sampled at **1-second intervals**.
- **Peak throughput** is the maximum 1-second throughput window in the first 30 seconds.
- **Steady-state throughput** is the mean throughput in the final 5 minutes.
- If a throttle reason is already active in the first sample, the notebook reports **“active from first sample; onset not observed”** instead of inventing an onset time.
- NVIDIA thermal slowdown is reported separately from power-cap or other throttle reasons.


In [ ]:
first30 = thermal[thermal["elapsed_s"] <= 30.0]
final5_start = max(0.0, float(thermal["elapsed_s"].max()) - 300.0)
final5 = thermal[thermal["elapsed_s"] >= final5_start]

if first30.empty or final5.empty:
    raise RuntimeError("Thermal run does not contain enough data for early/final windows.")

early_mean_tflops = float(first30["throughput_tflops"].mean())
peak_first30_tflops = float(first30["throughput_tflops"].max())
steady_tflops = float(final5["throughput_tflops"].mean())
steady_pct_peak = 100.0 * steady_tflops / peak_first30_tflops

def active_reason(v):
    s = str(v).strip().lower()
    return s in {"active", "yes", "true", "1"} or (
        s.startswith("0x") and s != "0x0000000000000000"
    )

reason_cols = [c for c in THROTTLE_FIELDS if c in thermal.columns]
specific_cols = [c for c in reason_cols if c != "clocks_throttle_reasons.active"]

active_mask = pd.Series(False, index=thermal.index)
if specific_cols:
    for c in specific_cols:
        active_mask = active_mask | thermal[c].map(active_reason)
elif "clocks_throttle_reasons.active" in thermal.columns:
    active_mask = thermal["clocks_throttle_reasons.active"].map(active_reason)

throttle_status = "none_observed"
throttle_onset = None
throttle_first_observed_s = None
throttle_reason = "none observed"
onset_temp = np.nan
onset_power = np.nan

if active_mask.any():
    first_active_idx = active_mask[active_mask].index[0]
    r = thermal.loc[first_active_idx]
    throttle_first_observed_s = float(r["elapsed_s"])
    onset_temp = float(r["temperature_C"])
    onset_power = float(r["power_draw_W"])

    reasons_at_row = [
        c for c in specific_cols
        if c in thermal.columns and active_reason(r[c])
    ]
    if not reasons_at_row and "clocks_throttle_reasons.active" in thermal.columns:
        if active_reason(r["clocks_throttle_reasons.active"]):
            reasons_at_row = ["clocks_throttle_reasons.active"]

    throttle_reason = ", ".join(reasons_at_row) if reasons_at_row else "NVIDIA active throttle reason"

    if first_active_idx == thermal.index[0]:
        throttle_status = "active_from_first_sample"
        throttle_onset = None
    else:
        throttle_status = "onset_observed"
        throttle_onset = throttle_first_observed_s

thermal_col = "clocks_throttle_reasons.hw_thermal_slowdown"
thermal_throttle_observed = (
    thermal_col in thermal.columns
    and thermal[thermal_col].map(active_reason).any()
)

part_e = {
    "uuid": GPU_UUID,
    "gpu": GPU_NAME,
    "duration_s": float(thermal["elapsed_s"].max()),
    "sample_count": int(len(thermal)),
    "early_first_30s_mean_tflops": early_mean_tflops,
    "peak_first_30s_tflops": peak_first30_tflops,
    "steady_final_5min_tflops": steady_tflops,
    "steady_state_over_peak_pct": steady_pct_peak,
    "throttle_status": throttle_status,
    "throttle_onset_s": throttle_onset,
    "throttle_first_observed_s": throttle_first_observed_s,
    "throttle_reason": throttle_reason,
    "throttle_first_observed_temp_C": None if np.isnan(onset_temp) else onset_temp,
    "throttle_first_observed_power_W": None if np.isnan(onset_power) else onset_power,
    "thermal_throttle_observed": bool(thermal_throttle_observed),
    "max_temperature_C": float(thermal["temperature_C"].max()),
    "max_power_draw_W": float(thermal["power_draw_W"].max()),
    "min_sm_clock_mhz": float(thermal["sm_clock_mhz"].min()),
    "max_sm_clock_mhz": float(thermal["sm_clock_mhz"].max()),
}

(ARTIFACT_DIR / "part_e_metrics.json").write_text(json.dumps(part_e, indent=2))
display(pd.DataFrame([part_e]))

log(
    f"Part E throughput | UUID={GPU_UUID} | early_mean={early_mean_tflops:.2f} TFLOPS | "
    f"first30_peak={peak_first30_tflops:.2f} TFLOPS | "
    f"final5min={steady_tflops:.2f} TFLOPS | steady/peak={steady_pct_peak:.2f}%"
)

if throttle_status == "active_from_first_sample":
    log(
        f"Part E throttle: active at the first sample t={throttle_first_observed_s:.1f}s; "
        f"onset was not observed | reason={throttle_reason} | "
        f"temp={onset_temp:.1f}C | power={onset_power:.1f}W"
    )
elif throttle_status == "onset_observed":
    log(
        f"Part E throttle onset observed at {throttle_onset:.1f}s | "
        f"reason={throttle_reason} | temp={onset_temp:.1f}C | power={onset_power:.1f}W"
    )
else:
    log("Part E throttle: none observed")

if not thermal_throttle_observed:
    log(
        f"No NVIDIA thermal-slowdown reason was observed; "
        f"maximum temperature was {part_e['max_temperature_C']:.1f}C."
    )


# Part F — assemble every required measurement

This cell reads the saved UUID-labeled measurements from Parts B–E and writes `METRICS.md`.

The summary table follows the assignment's exact requested rows:
- Peak achieved TFLOPS (BF16)
- % of theoretical peak (BF16)
- Effective bandwidth
- Naive attention OOM length
- Fused attention OOM length
- Steady-state / peak throughput
- Throttle onset

In [ ]:
# Load prior measurements.
part_b = pd.read_csv(ARTIFACT_DIR / "part_b_precision_results.csv")
part_c = json.loads((ARTIFACT_DIR / "part_c_metrics.json").read_text())
boundaries = json.loads((ARTIFACT_DIR / "part_d_boundaries.json").read_text())
fit_info = json.loads((ARTIFACT_DIR / "part_d_quadratic_fit.json").read_text())
speed = pd.read_csv(ARTIFACT_DIR / "part_d_speedups.csv")
part_e = json.loads((ARTIFACT_DIR / "part_e_metrics.json").read_text())
plateaus = pd.read_csv(ARTIFACT_DIR / "part_b_plateaus.csv")

bf16 = part_b[
    (part_b["precision"] == "BF16")
    & part_b["achieved_tflops"].notna()
].copy()

if bf16.empty:
    raise RuntimeError(
        "No successful BF16 measurement found. "
        "Do not substitute FP16; rerun Part B on the required RTX 4090/5090."
    )

peak_bf16_row = bf16.loc[bf16["achieved_tflops"].idxmax()]

def boundary_text(b):
    if b.get("smallest_failure") is None:
        return (
            f">= {b.get('largest_success')} tested successfully; "
            "no failure through explicit search cap"
        )
    return (
        f"largest success {b.get('largest_success')}; "
        f"smallest failure {b.get('smallest_failure')} "
        f"(test resolution {b.get('step')})"
    )

status = part_e.get("throttle_status", "none_observed")
if status == "active_from_first_sample":
    throttle_text = (
        f"active from first sample (t={part_e.get('throttle_first_observed_s'):.1f} s); "
        "onset not observed"
    )
elif status == "onset_observed":
    throttle_text = f"{part_e['throttle_onset_s']:.1f} s ({part_e.get('throttle_reason', '')})"
else:
    throttle_text = "none observed"

summary = pd.DataFrame([
    {
        "Measurement": "Peak achieved TFLOPS (BF16)",
        "Your GPU": f"{peak_bf16_row['achieved_tflops']:.3f}",
        "Notes": f"N={int(peak_bf16_row['N'])}, UUID={GPU_UUID}",
    },
    {
        "Measurement": "% of theoretical peak (BF16)",
        "Your GPU": f"{peak_bf16_row['pct_theoretical_peak']:.2f}%",
        "Notes": f"dense theoretical peak={peak_bf16_row['theoretical_tflops']:.1f} TFLOPS",
    },
    {
        "Measurement": "Effective bandwidth (GB/s)",
        "Your GPU": f"{part_c['effective_bandwidth_GBs']:.2f}",
        "Notes": f"{part_c['bandwidth_pct_specified']:.2f}% of {part_c['specified_bandwidth_GBs']:.0f} GB/s",
    },
    {
        "Measurement": "Naive attention OOM length",
        "Your GPU": boundary_text(boundaries["naive"]),
        "Notes": f"B={boundaries['batch']}, H={boundaries['heads']}, D={boundaries['head_dim']}",
    },
    {
        "Measurement": "Fused attention OOM length",
        "Your GPU": boundary_text(boundaries["fused"]),
        "Notes": boundaries["fused_backend"],
    },
    {
        "Measurement": "Steady-state / peak throughput",
        "Your GPU": f"{part_e['steady_state_over_peak_pct']:.2f}%",
        "Notes": "final 5 min mean / maximum 1-second window in first 30 s",
    },
    {
        "Measurement": "Throttle onset (s, or none)",
        "Your GPU": throttle_text,
        "Notes": f"thermal throttle observed={part_e.get('thermal_throttle_observed')}",
    },
])

display(summary)

metrics_lines = [
    "# HW2.5 METRICS",
    "",
    f"- GPU: {GPU_NAME}",
    f"- UUID: `{GPU_UUID}`",
    f"- NVIDIA architecture: {CARD_SPECS['architecture']}",
    f"- Memory: {CARD_SPECS['memory_type']}",
    f"- Specified memory bandwidth: {CARD_SPECS['memory_bandwidth_GBs']} GB/s",
    f"- Tensor cores: {CARD_SPECS['tensor_core_generation']}",
    f"- Driver-supported CUDA version: {GPU_INFO.get('driver_cuda_version')}",
    f"- PyTorch CUDA build: {GPU_INFO.get('pytorch_cuda_version')}",
    f"- NVIDIA source: {CARD_SPECS['source_url']}",
    f"- NVIDIA source accessed: {CARD_SPECS.get('source_accessed_utc', '2026-09-17')}",
    "",
    "## Table HW2.5.1 — Summary",
    "",
    summary.to_markdown(index=False),
    "",
    "## Part B — Precision benchmark",
    "",
    part_b.to_markdown(index=False),
    "",
    "## Part B — Operational plateau",
    "",
    "Criterion: the first tested N whose complete larger-N tail has throughput spread <=10% "
    "and CV <=5% at every point. If no tail qualifies, no plateau is claimed.",
    "",
    plateaus.to_markdown(index=False),
    "",
    "## Part C — Roofline measurements",
    "",
    pd.DataFrame([part_c]).to_markdown(index=False),
    "",
    "## Part D — Attention",
    "",
    f"- Naive OOM bracket: {boundary_text(boundaries['naive'])}",
    f"- Fused backend: {boundaries['fused_backend']}",
    f"- Fused search result: {boundary_text(boundaries['fused'])}",
    f"- Fused interpretation: {boundaries.get('fused_interpretation', '')}",
    f"- Measured naive quadratic coefficient: `{fit_info['quadratic_coefficient_GiB_per_token2']:.12e}` GiB/token²",
    f"- Closed-form quadratic coefficient: `{fit_info['theoretical_quadratic_coefficient_GiB_per_token2']:.12e}` GiB/token²",
    f"- Measured/theory ratio: `{fit_info['quadratic_coefficient_ratio_measured_to_theory']:.6f}`",
    f"- Coefficient error: `{fit_info['quadratic_coefficient_error_pct']:.3f}%`",
    f"- Fit: `{fit_info['fit_expression']}`",
    "",
    "The measured S² coefficient is checked against the closed-form storage cost of the "
    "simultaneously-live score and softmax matrices: 2 × B × H × sizeof(dtype) / 2³⁰ GiB/token².",
    "",
    "### Fused speedup",
    "",
    speed.to_markdown(index=False) if not speed.empty else "No common successful lengths to compute speedup.",
    "",
    "The fused kernel avoids storing the full S×S attention intermediate. A very-large-S fused "
    "failure can instead be dominated by Q/K/V/output storage and workspace, so the fused search "
    "cap is reported explicitly rather than presented as a pure attention limit.",
    "",
    "## Part E — sustained load",
    "",
    pd.DataFrame([part_e]).to_markdown(index=False),
    "",
]

metrics_path = ARTIFACT_DIR / "METRICS.md"
metrics_path.write_text("\n".join(metrics_lines))
log(f"Wrote {metrics_path}")


Final Validation Checklist

In [ ]:
required_patterns = {
    "nvidia-smi basic": [ARTIFACT_DIR / "nvidia_smi_basic.txt"],
    "nvidia-smi -q": list(ARTIFACT_DIR.glob("nvidia_smi_q_gpu*.txt")),
    "environment.json": [ARTIFACT_DIR / "environment.json"],
    "pip_freeze.txt": [ARTIFACT_DIR / "pip_freeze.txt"],
    "thermal log": [ARTIFACT_DIR / "thermal_log.csv"],
    "METRICS.md": [ARTIFACT_DIR / "METRICS.md"],
    "RUN_LOG.txt": [ARTIFACT_DIR / "RUN_LOG.txt"],
    "reservation/GPU-hours": [ARTIFACT_DIR / "reservation_gpu_hours.md"],
    "Part B CSV": [ARTIFACT_DIR / "part_b_precision_results.csv"],
    "Part C JSON": [ARTIFACT_DIR / "part_c_metrics.json"],
    "Part D boundaries": [ARTIFACT_DIR / "part_d_boundaries.json"],
    "Part E JSON": [ARTIFACT_DIR / "part_e_metrics.json"],
}

checks = []

for label, paths in required_patterns.items():
    exists = bool(paths) and all(Path(p).exists() for p in paths)

    checks.append({
        "deliverable": label,
        "exists": exists,
        "paths": ", ".join(str(p) for p in paths) if paths else "MISSING",
    })

# Reservation content check
reservation_path = ARTIFACT_DIR / "reservation_gpu_hours.md"
reservation_text = reservation_path.read_text() if reservation_path.exists() else ""

reservation_complete = (
    reservation_path.exists()
    and "[FILL IN]" not in reservation_text
)

checks.append({
    "deliverable": "reservation fields completed",
    "exists": reservation_complete,
    "paths": str(reservation_path),
})

# GPU check
gpu_ok = bool(GPU_INFO.get("hw25_supported_gpu", False)) and (
    "RTX 4090" in GPU_NAME or "RTX 5090" in GPU_NAME
)

checks.append({
    "deliverable": "required RTX 4090/5090 hardware",
    "exists": gpu_ok,
    "paths": GPU_NAME,
})

# Precision check
part_b_check = pd.read_csv(
    ARTIFACT_DIR / "part_b_precision_results.csv"
)

required_precisions = {
    "FP32",
    "TF32",
    "FP16",
    "BF16"
}

successful_precisions = set(
    part_b_check.loc[
        part_b_check["achieved_tflops"].notna(),
        "precision"
    ]
)

precision_ok = required_precisions.issubset(
    successful_precisions
)

checks.append({
    "deliverable": "FP32/TF32/FP16/BF16 measurements",
    "exists": precision_ok,
    "paths": ", ".join(sorted(successful_precisions)),
})

check_df = pd.DataFrame(checks)
display(check_df)

missing = check_df.loc[
    ~check_df["exists"],
    "deliverable"
].tolist()

if missing:
    raise RuntimeError(
        "Final validation failed: " + ", ".join(missing)
    )

figs = sorted(FIG_DIR.glob("*.png"))

log(f"Figures found ({len(figs)}):")

for f in figs:
    log("  " + str(f))

# Git tag check
git_tag_ok = False

git_probe = subprocess.run(
    [
        "git",
        "-C",
        str(ASSIGNMENT_DIR),
        "tag",
        "--list",
        "hw2-5"
    ],
    capture_output=True,
    text=True,
)

if git_probe.returncode == 0:
    git_tag_ok = git_probe.stdout.strip() == "hw2-5"

if git_tag_ok:
    log("Git tag hw2-5 found.")
else:
    log(
        "WARNING: Git tag hw2-5 was not verified here. "
        "Create it in the submitted repository."
    )

# Create ZIP
import shutil

archive_path = str(
    ASSIGNMENT_DIR / "hw2_5_artifacts"
)

archive = shutil.make_archive(
    archive_path,
    "zip",
    root_dir=ARTIFACT_DIR,
)

log(
    f"Created artifact backup archive: {archive}"
)